# Path Queries with Stardog (LINDAS)

In [46]:
import json
import pprint
import pandas as pd
from pyodide.ffi import to_js
from IPython.display import JSON, HTML
from js import Object, fetch
from io import StringIO

async def path(query_string, address = "https://ld.admin.ch/query"):
    
    # try the Post request with help of JS fetch
    # the creation of the request header is a little bit complicated because it needs to be a 
    # JavaScript JSON that is made within a Python source code
    try:
        resp = await fetch(address,
          method="POST",
          body="query=" + query_string,
          credentials="same-origin",
          headers=Object.fromEntries(to_js({"Content-Type": "application/x-www-form-urlencoded; charset=UTF-8", 
                                            "Accept": "application/sparql-results+json" })),
        )
    except:
        raise RuntimeError("fetch failed")
    
    
    if resp.ok:
        res = await resp.text()
        # ld.admin.ch throws errors starting with '{"message":'
        if '{"message":' in res:
            error = json.loads(res)
            raise RuntimeError("SPARQL query malformed: " + error["message"])
        # geo.ld.admin.ch throws errors starting with 'Parse error:'
        elif 'Parse error:' in res:
            raise RuntimeError("SPARQL query malformed: " + res)
        else:
            # if everything works out, create a pandas dataframe from the csv result
            # df = pd.read_csv(StringIO(res))
            return res
    else:
        # fedlex.data.admin.ch throws error with response status 400
        if resp.status == 400:
            raise RuntimeError("Response status 400: Possible malformed SPARQL query. No syntactic advice available.")
        else:
            raise RuntimeError("Response status " + str(resp.status))
            
            


In [52]:
res = await path("""

PATHS 
START ?x = <https://ld.admin.ch/municipality/version/14062> 
END ?y 
VIA <https://version.link/successor> | <https://version.link/predecessor>


""", "https://test.lindas.admin.ch/query")

res = json.loads(res)
pprint.pprint(res["results"]["bindings"])

[{'x': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/14062'},
  'y': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/11930'}},
 {},
 {'x': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/14062'},
  'y': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/12041'}},
 {},
 {'x': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/14062'},
  'y': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/12401'}},
 {},
 {'x': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/14062'},
  'y': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/13753'}},
 {},
 {'x': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/14062'},
  'y': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/14063'}},
 {},
 {'x': {'type': 'uri',
        'value': 'https://ld.admin.ch

In [61]:
res = await path("""

PREFIX vl: <https://version.link/>
PREFIX schema: <http://schema.org/>

PATHS 
START ?x = <https://ld.admin.ch/municipality/version/14062>
END ?y 
VIA {

#BIND(vl:successor as ?p)

VALUES ?p {
    vl:successor
    vl:predecessor
}

?x ?p ?y.

?x schema:name ?xName.
?y schema:name ?yName.
}


""", "https://test.lindas.admin.ch/query")

res = json.loads(res)
pprint.pprint(res["results"]["bindings"])

[{'p': {'type': 'uri', 'value': 'https://version.link/predecessor'},
  'x': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/14062'},
  'xName': {'type': 'literal', 'value': 'Berg (TG)'},
  'y': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/11930'},
  'yName': {'type': 'literal', 'value': 'Guntershausen b. Birwin.'}},
 {},
 {'p': {'type': 'uri', 'value': 'https://version.link/predecessor'},
  'x': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/14062'},
  'xName': {'type': 'literal', 'value': 'Berg (TG)'},
  'y': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/12041'},
  'yName': {'type': 'literal', 'value': 'Graltshausen'}},
 {},
 {'p': {'type': 'uri', 'value': 'https://version.link/predecessor'},
  'x': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/14062'},
  'xName': {'type': 'literal', 'value': 'Berg (TG)'},
  'y': {'type': 'uri',
        'value'